# 04 Temporada 2025 — arquetipo modal

Lee `results/2025/season/` generado por `python scripts/run_pipeline.py --season 2025`. No descarga sesiones.

La tendencia de temporada es el arquetipo modal: el nombre de clase que más se repite entre los Grandes Premios. `stability.csv` es la fracción de carreras que coinciden con esa moda. No hay un segundo clustering y no se arma una mediana de vectores por GP. Ver `docs/mapeo_ptf.md`.

`build_season_driver_matrix` en `src/f1_style_clustering/season.py` sigue calculando la mediana de los vectores por GP, y los tests la cubren. `run_season_pipeline` no la llama y no escribe `driver_matrix.csv`. Este notebook no la lee. Los archivos de la temporada son `driver_summary.csv`, `driver_profiles.csv`, `archetype_by_race.csv` y `stability.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
season_dir = ROOT / "results" / "2025" / "season"
needed = ["driver_summary.csv", "archetype_by_race.csv", "stability.csv"]
missing = [name for name in needed if not (season_dir / name).exists()]
print(season_dir, "ok" if not missing else "MISSING — python scripts/run_pipeline.py --season 2025")
if missing:
    raise FileNotFoundError(
        "Faltan "
        + ", ".join(missing)
        + ". No se lee driver_matrix.csv: la temporada no es una mediana de vectores ni un segundo clustering."
    )


`driver_summary.csv` es un piloto por fila. `Archetype` es la moda de sus arquetipos de GP. `n_races` es cuántos GP entraron. No hay columna `Cluster` de un clustering de temporada.

In [ ]:
summary = pd.read_csv(season_dir / "driver_summary.csv", index_col=0)
stability = pd.read_csv(season_dir / "stability.csv", index_col=0)
by_race = pd.read_csv(season_dir / "archetype_by_race.csv", index_col=0)
summary


Cada celda es el arquetipo de ese piloto en ese GP. `agreement` es la fracción de carreras iguales al arquetipo modal de temporada.

In [ ]:
plt.figure(figsize=(14, 8))
heat = by_race.copy()
cats = sorted({v for v in heat.values.ravel() if pd.notna(v)})
codes = heat.apply(lambda col: col.map({c: i for i, c in enumerate(cats)}))
ax = sns.heatmap(codes, cmap="tab10", cbar=False, linewidths=0.2)
ax.set_title("Arquetipo modal por GP")
plt.tight_layout()
plt.show()
stability.sort_values("agreement", ascending=False)
